In [1]:
!pip install pandas scikit-learn

In [2]:
import pandas as pd
from urllib.error import HTTPError

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

In [3]:
SHEET_ID = "1P9KCHIZn5ZxX4A2Cj4wTJrzeeOWtbgqksubfC9GFeMo"
GID = "0"  # troque se necessário

url_export_csv = f"https://docs.google.com/spreadsheets/d/{SHEET_ID}/export?format=csv&gid={GID}"
url_gviz_csv   = f"https://docs.google.com/spreadsheets/d/{SHEET_ID}/gviz/tq?tqx=out:csv&gid={GID}"

def ler_google_sheets():
    try:
        print("Tentando via /export ...")
        return pd.read_csv(url_export_csv)
    except HTTPError as e:
        print("Erro export:", e)

    try:
        print("Tentando via GVIZ ...")
        return pd.read_csv(url_gviz_csv)
    except HTTPError as e:
        print("Erro GVIZ:", e)

    raise RuntimeError("Não consegui acessar a planilha. Verifique o compartilhamento.")

df = ler_google_sheets()

print("\nColunas:", list(df.columns))
df.head()

Tentando via /export ...
Erro export: HTTP Error 400: Bad Request
Tentando via GVIZ ...

Colunas: ['time', 'ano', 'jogos', 'vitorias', 'empates', 'derrotas', 'gols_pro', 'gols_contra', 'pontos', 'classificado_top6']


,time,ano,jogos,vitorias,empates,derrotas,gols_pro,gols_contra,pontos,classificado_top6
0,Flamengo,2021,38,11,3,24,72,37,36,0
1,Corinthians,2021,38,15,6,17,33,67,51,0
2,Palmeiras,2021,38,11,13,14,72,77,46,0
3,São Paulo,2021,38,25,4,9,62,47,79,1
4,Santos,2021,38,9,3,26,30,33,30,0


In [4]:
target_col = "classificado_top6"

if target_col not in df.columns:
    raise ValueError(f"❌ Coluna '{target_col}' não encontrada!")

X_raw = df.drop(columns=[target_col]).copy()
y = pd.to_numeric(df[target_col], errors="coerce")

# mantém apenas colunas numéricas
X = X_raw.select_dtypes(include="number").copy()
X = X.apply(pd.to_numeric, errors="coerce")

# remove linhas inválidas
mask = X.notna().all(axis=1) & y.notna()
X = X.loc[mask]
y = y.loc[mask].astype(int)

print("Linhas finais:", len(X))
print("Features:", list(X.columns))
print("Classes:", y.value_counts().to_dict())

Linhas finais: 45
Features: ['ano', 'jogos', 'vitorias', 'empates', 'derrotas', 'gols_pro', 'gols_contra', 'pontos']
Classes: {1: 23, 0: 22}


In [5]:
def avaliar(nome, model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)

    print("\n==============================")
    print(nome)
    print(f"Acurácia: {acc:.4f}")
    print("Matriz de confusão:\n", confusion_matrix(y_test, y_pred))
    print("Relatório:\n", classification_report(y_test, y_pred, digits=4))

    return acc

In [6]:
test_sizes = [0.2, 0.3]
k_values = [3, 5, 7, 9]
depth_values = [3, 5, 10, None]

feature_sets = {
    "todas": list(X.columns),
    "sem_ano": [c for c in X.columns if c != "ano"],
    "basico": [c for c in ["pontos", "gols_pro", "gols_contra"] if c in X.columns],
    "resultado": [c for c in ["vitorias", "empates", "derrotas", "pontos"] if c in X.columns],
}

In [7]:
resultados = []

for test_size in test_sizes:
    for fs_name, cols in feature_sets.items():
        if len(cols) == 0:
            continue

        X_fs = X[cols].copy()
        strat = y if y.nunique() > 1 else None

        X_train, X_test, y_train, y_test = train_test_split(
            X_fs, y, test_size=test_size, random_state=42, stratify=strat
        )

        # KNN
        for k in k_values:
            knn = Pipeline([
                ("scaler", StandardScaler()),
                ("model", KNeighborsClassifier(n_neighbors=k))
            ])

            acc = avaliar(
                f"KNN | k={k} | split={int((1-test_size)*100)}/{int(test_size*100)} | {fs_name}",
                knn, X_train, X_test, y_train, y_test
            )

            resultados.append(("KNN", fs_name, test_size, f"k={k}", acc))

        # Árvore
        for depth in depth_values:
            tree = DecisionTreeClassifier(max_depth=depth, random_state=42)

            acc = avaliar(
                f"Árvore | depth={depth} | split={int((1-test_size)*100)}/{int(test_size*100)} | {fs_name}",
                tree, X_train, X_test, y_train, y_test
            )

            resultados.append(("Arvore", fs_name, test_size, f"depth={depth}", acc))


KNN | k=3 | split=80/20 | todas
Acurácia: 0.8889
Matriz de confusão:
 [[3 1]
 [0 5]]
Relatório:
               precision    recall  f1-score   support

           0     1.0000    0.7500    0.8571         4
           1     0.8333    1.0000    0.9091         5

    accuracy                         0.8889         9
   macro avg     0.9167    0.8750    0.8831         9
weighted avg     0.9074    0.8889    0.8860         9


KNN | k=5 | split=80/20 | todas
Acurácia: 0.8889
Matriz de confusão:
 [[3 1]
 [0 5]]
Relatório:
               precision    recall  f1-score   support

           0     1.0000    0.7500    0.8571         4
           1     0.8333    1.0000    0.9091         5

    accuracy                         0.8889         9
   macro avg     0.9167    0.8750    0.8831         9
weighted avg     0.9074    0.8889    0.8860         9


KNN | k=7 | split=80/20 | todas
Acurácia: 0.8889
Matriz de confusão:
 [[3 1]
 [0 5]]
Relatório:
               precision    recall  f1-score   suppor

In [8]:
res_df = pd.DataFrame(resultados, columns=["modelo", "features", "test_size", "params", "acuracia"])
res_df = res_df.sort_values("acuracia", ascending=False)

print("\n===== TOP 10 =====")
res_df.head(10)


===== TOP 10 =====


,modelo,features,test_size,params,acuracia
7,Arvore,todas,0.2,depth=None,1.0
6,Arvore,todas,0.2,depth=10,1.0
5,Arvore,todas,0.2,depth=5,1.0
4,Arvore,todas,0.2,depth=3,1.0
12,Arvore,sem_ano,0.2,depth=3,1.0
13,Arvore,sem_ano,0.2,depth=5,1.0
14,Arvore,sem_ano,0.2,depth=10,1.0
15,Arvore,sem_ano,0.2,depth=None,1.0
23,Arvore,basico,0.2,depth=None,1.0
22,Arvore,basico,0.2,depth=10,1.0
